# USA Housing EDA

Exploratory data analysis for the ingested housing dataset used by the pipeline.

Dataset path: `data/raw/usa-housing-dataset/USA Housing Dataset.csv`

## Goals

- verify the schema and row counts
- inspect nulls, duplicates, and type expectations
- surface data quality issues before transformation and training
- translate EDA findings into validation rules

In [ ]:
from __future__ import annotations

import csv
from collections import Counter
from pathlib import Path
from statistics import mean

def is_repo_root(candidate: Path) -> bool:
    return (candidate / 'main.py').exists() and (candidate / 'src').exists() and (candidate / 'data').exists()

def find_repo_root(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    direct_candidates += [
        Path.home(),
        Path.home() / 'pranav',
        Path.home() / 'pranav' / 'house-price-ml',
        Path.home() / 'house-price-ml',
    ]

    seen = set()
    for candidate in direct_candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if is_repo_root(candidate):
            return candidate

    search_roots = [Path.home(), Path.home() / 'pranav']
    for search_root in search_roots:
        if not search_root.exists():
            continue
        for candidate in search_root.rglob('main.py'):
            repo_root = candidate.parent.resolve()
            if is_repo_root(repo_root):
                return repo_root

    raise FileNotFoundError('Could not locate repo root from current notebook kernel path.')

REPO_ROOT = find_repo_root(Path.cwd())
DATA_PATH = REPO_ROOT / 'data' / 'raw' / 'usa-housing-dataset' / 'USA Housing Dataset.csv'

print('Kernel cwd:', Path.cwd())
print('Repo root:', REPO_ROOT)
print('Dataset path:', DATA_PATH)

assert DATA_PATH.exists(), f'Missing dataset file: {DATA_PATH}'

with DATA_PATH.open('r', encoding='utf-8', newline='') as file_obj:
    reader = csv.DictReader(file_obj)
    rows = list(reader)
    columns = reader.fieldnames or []

len(rows), columns


In [ ]:
rows[:2]

In [ ]:
missing_counts = {column: sum(1 for row in rows if not row[column].strip()) for column in columns}
duplicate_rows = len(rows) - len({tuple((column, row[column]) for column in columns) for row in rows})
missing_counts, duplicate_rows

In [ ]:
numeric_columns = [
    'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
    'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement',
    'yr_built', 'yr_renovated'
]

numeric_summary = {}
for column in numeric_columns:
    values = [float(row[column]) for row in rows]
    numeric_summary[column] = {
        'min': min(values),
        'mean': round(mean(values), 2),
        'max': max(values),
    }

numeric_summary

In [ ]:
zero_or_negative_price_rows = [row for row in rows if float(row['price']) <= 0]
len(zero_or_negative_price_rows), zero_or_negative_price_rows[:3]

In [ ]:
country_counts = Counter(row['country'] for row in rows)
city_counts = Counter(row['city'] for row in rows).most_common(10)
state_prefix_counts = Counter(row['statezip'].split()[0] for row in rows).most_common(10)
country_counts, city_counts, state_prefix_counts

## EDA Findings To Feed The Pipeline

- The dataset has `4140` rows and `18` columns.
- No missing values were found in the current download.
- No exact duplicate rows were found.
- `country` is constant as `USA`; `statezip` values are all prefixed with `WA`.
- `price` contains `49` non-positive values, which should be treated as a data quality warning and handled before model training.
- `sqft_living`, `sqft_lot`, `sqft_above`, and `yr_built` are expected to remain strictly positive.
- The `date` column follows the `%Y-%m-%d %H:%M:%S` format and should be validated as such.